# Notebook 12 — Assembling a Production sklearn Pipeline

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 70 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Build a production-grade `sklearn` `Pipeline` with custom transformers
2. Implement `BaseEstimator` + `TransformerMixin` subclasses from scratch
3. Use `ColumnTransformer` to handle mixed-type (numeric + categorical + text) data
4. Tune pipeline hyperparameters with the double-underscore `__` syntax
5. Serialize and reload a complete pipeline for production inference

**Week 12 Theme:** Airbnb-style listing price prediction


## The Assembly-Line Analogy

Think of a **sklearn Pipeline** as a **factory assembly line with quality checkpoints**.

- **Station 1 — Raw material intake (Imputation):** Workers inspect incoming parts, fill any missing pieces with the median part from yesterday's supply. They only look at yesterday's supply — never today's customer order.
- **Station 2 — Shaping (Scaling):** A press machine squeezes every part to a standard size. The calibration is set once using yesterday's batch — applying that same calibration to today's batch is fine, because the machine never "saw" today.
- **Station 3 — Surface coating (Encoding):** A paint robot coats each part based on its category. Unknown categories get a neutral coat (handle_unknown='ignore').
- **Final station — Assembly (Model):** The model combines all prepared parts into a finished product (prediction).

**The line is reproducible and restartable (serialization).** Stamp a QR code on the whole line, box it up with `joblib.dump`, and ship it to any factory floor worldwide — `joblib.load` reassembles the identical line.

**Without a Pipeline, workers share tools across batches.** If you fit `StandardScaler` on the entire dataset before cross-validation, the "test fold" has already influenced the scaler's calibration. That is data leakage — your quality inspector is grading the parts she helped make.


## Why Does This Matter for ML?

| Problem | Without Pipeline | With Pipeline |
|---------|-----------------|---------------|
| **Data leakage** | Easy to accidentally fit scaler on all data before CV | Impossible — each CV fold fits its own scaler |
| **Reproducibility** | Manual sequence of steps, easy to miss one | Single object encapsulates every step |
| **Deployment** | Must recreate preprocessing logic in production separately | One `joblib.load` + `.predict()` call |
| **Hyperparameter search** | Must manually rebuild each variant | `GridSearchCV` over `pipe__step__param` syntax |
| **Column tracking** | Lost after numpy transformations | `get_feature_names_out()` traces every column |

**Bottom line:** A Pipeline is not a convenience — it is a correctness requirement for any ML system that goes to production.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

sns.set_theme(style='whitegrid')
np.random.seed(42)

# ── Generate a synthetic Airbnb-style dataset with mixed column types ──
N = 1000

# Numeric features
bedrooms       = np.random.choice([1, 2, 3, 4, 5], N, p=[0.3, 0.35, 0.2, 0.1, 0.05])
bathrooms      = np.clip(bedrooms * 0.6 + np.random.normal(0, 0.3, N), 0.5, 5.0).round(1)
accommodates   = np.clip(bedrooms * 1.8 + np.random.randint(-1, 2, N), 1, 16).astype(int)
review_score   = np.clip(np.random.normal(4.5, 0.4, N), 1.0, 5.0).round(2)
num_reviews    = np.random.negative_binomial(5, 0.3, N).astype(float)

# Categorical features
neighbourhood_group = np.random.choice(
    ['Central', 'East', 'West', 'North', 'South'], N,
    p=[0.25, 0.2, 0.2, 0.18, 0.17]
)
room_type = np.random.choice(
    ['Entire home/apt', 'Private room', 'Shared room'], N,
    p=[0.55, 0.38, 0.07]
)

# Text feature (description)
descriptions = [
    'cosy studio near transport links',
    'beautiful apartment with panoramic city views',
    'budget room shared bathroom close to centre',
    'modern flat fully equipped kitchen wifi',
    'luxury penthouse rooftop terrace skyline',
    'quiet neighbourhood close to parks family friendly',
    'professional workspace fast wifi conference room',
    'charming Victorian house private garden',
]
description = np.random.choice(descriptions, N)

# Boolean feature
is_superhost = np.random.choice([True, False], N, p=[0.35, 0.65])

# Construct price (log scale target) with realistic drivers
location_bonus = np.where(neighbourhood_group == 'Central', 0.4, 0.0)
room_bonus     = np.where(room_type == 'Entire home/apt', 0.5,
                 np.where(room_type == 'Private room', 0.1, -0.3))
log_price = (
    3.5
    + 0.25 * bedrooms
    + 0.1  * review_score
    + 0.02 * np.log1p(num_reviews)
    + location_bonus
    + room_bonus
    + np.random.normal(0, 0.2, N)   # noise
)

df = pd.DataFrame({
    'bedrooms':            bedrooms.astype(float),
    'bathrooms':           bathrooms,
    'accommodates':        accommodates.astype(float),
    'review_score':        review_score,
    'num_reviews':         num_reviews,
    'neighbourhood_group': neighbourhood_group,
    'room_type':           room_type,
    'description':         description,
    'is_superhost':        is_superhost.astype(int),  # bool → int for sklearn
    'log_price':           log_price,
})

# Inject ~5% missing values into numeric columns
for col in ['bedrooms', 'bathrooms', 'review_score', 'num_reviews']:
    mask = np.random.rand(N) < 0.05
    df.loc[mask, col] = np.nan

X = df.drop(columns='log_price')
y = df['log_price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set : {X_train.shape}")
print(f"Test set     : {X_test.shape}")
print(f"\nColumn dtypes:\n{X.dtypes}")
print(f"\nMissing value rates:\n{X.isnull().mean().round(3)}")

In [ ]:
np.random.seed(42)

# ── EDA: understand the dataset before building the pipeline ──
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (a) log_price distribution
ax = axes[0, 0]
ax.hist(df['log_price'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax.set_xlabel('log(price)')
ax.set_title('(a) Target Distribution')

# (b) log_price by room type
ax = axes[0, 1]
for rt, colour in zip(['Entire home/apt', 'Private room', 'Shared room'],
                      ['steelblue', 'darkorange', 'tomato']):
    vals = df.loc[df['room_type'] == rt, 'log_price']
    ax.hist(vals, bins=30, alpha=0.55, label=rt, color=colour, edgecolor='white')
ax.set_xlabel('log(price)')
ax.set_title('(b) Price by Room Type')
ax.legend(fontsize=8)

# (c) Correlation heatmap of numeric columns vs target
ax = axes[1, 0]
num_cols = ['bedrooms', 'bathrooms', 'accommodates', 'review_score', 'num_reviews', 'is_superhost', 'log_price']
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('(c) Feature Correlations')

# (d) Missing value heatmap
ax = axes[1, 1]
missing_pct = df.isnull().mean() * 100
missing_pct = missing_pct[missing_pct > 0]  # only show columns with missing data
ax.barh(missing_pct.index, missing_pct.values, color='tomato', edgecolor='white')
ax.set_xlabel('Missing (%)')
ax.set_title('(d) Missing Value Rates')

plt.suptitle('Airbnb Dataset — Exploratory Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## The Leakage Problem: Why Pipeline CV ≠ Scale-then-CV

Imagine 5-fold cross-validation. Your dataset has 1000 rows. In fold 1:
- Training folds: rows 200–1000 (800 rows)
- Validation fold: rows 0–199 (200 rows)

**Wrong approach (leaky):** Fit `StandardScaler` on ALL 1000 rows first. The scaler has already "seen" the 200 validation rows. When you evaluate on those 200 rows, your model has an unfair advantage — it knows their scale. Your CV score is optimistic.

**Correct approach (Pipeline):** The scaler is fitted only on the 800 training rows inside each fold. The 200 validation rows are transformed using the 800-row statistics — exactly what will happen in production when unseen data arrives.

The difference is usually small for StandardScaler but can be large for:
- Target encoding (huge leakage potential)
- Imputation with learned statistics
- Feature selection using target correlation


In [ ]:
np.random.seed(42)

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# Use only numeric features for this leakage demo
numeric_cols_demo = ['bedrooms', 'bathrooms', 'accommodates', 'review_score',
                     'num_reviews', 'is_superhost']

X_num = X[numeric_cols_demo].fillna(X[numeric_cols_demo].median())
X_num_arr = X_num.values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ── LEAKY approach: scale the entire dataset BEFORE CV ──
scaler_leaky = StandardScaler()
X_scaled_all = scaler_leaky.fit_transform(X_num_arr)  # sees ALL rows

leaky_scores = []
for train_idx, val_idx in kf.split(X_scaled_all):
    model = Ridge(alpha=1.0)
    model.fit(X_scaled_all[train_idx], y.values[train_idx])
    pred = model.predict(X_scaled_all[val_idx])
    leaky_scores.append(r2_score(y.values[val_idx], pred))

# ── CORRECT approach: Pipeline ensures scaler is fitted only on training folds ──
pipe_correct = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   Ridge(alpha=1.0)),
])
correct_scores = cross_val_score(
    pipe_correct, X_num_arr, y.values, cv=kf, scoring='r2'
)

print("=== Leakage Demo ===")
print(f"LEAKY  CV R² (scale-then-CV): {np.mean(leaky_scores):.4f} ± {np.std(leaky_scores):.4f}")
print(f"CORRECT Pipeline CV R²      : {correct_scores.mean():.4f} ± {correct_scores.std():.4f}")
print()
print("Even a small upward bias in CV score leads to poor generalisation estimates.")
print("With stronger preprocessing (e.g. target encoding) the gap is much larger.")

In [ ]:
np.random.seed(42)

# ── Minimal two-step Pipeline: scale → ridge ──
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

numeric_cols_demo = ['bedrooms', 'bathrooms', 'accommodates', 'review_score',
                     'num_reviews', 'is_superhost']

X_num_train = X_train[numeric_cols_demo].values
X_num_test  = X_test[numeric_cols_demo].values

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   Ridge(alpha=1.0)),
])

pipe.fit(X_num_train, y_train)   # all steps fitted in one call

# Display the pipeline tree — shows nested structure clearly
print(pipe)
print()

# Access individual steps by name
print("Scaler mean (first 3 features):", pipe.named_steps['scaler'].mean_[:3].round(3))

# Predict flows through every step automatically
preds = pipe.predict(X_num_test)
print(f"Test R² (minimal pipeline): {r2_score(y_test, preds):.3f}")

## Custom Transformers: `BaseEstimator` + `TransformerMixin`

Sometimes you need a preprocessing step that sklearn does not ship. You can write your own transformer that plugs seamlessly into any Pipeline.

**The contract has three rules:**
1. Inherit from both `BaseEstimator` and `TransformerMixin`
2. `fit(X, y=None)` — learn what you need from training data, return `self`
3. `transform(X)` — apply the learned transformation, return array or DataFrame

`BaseEstimator` gives you `get_params()` and `set_params()` for free (required by `GridSearchCV`).  
`TransformerMixin` gives you `fit_transform()` for free (calls `fit` then `transform`).

**Plain English rule:** Store only what you must learn in `fit`. Keep `transform` pure — same input always produces same output given the same fitted state.


In [ ]:
np.random.seed(42)

from sklearn.base import BaseEstimator, TransformerMixin, clone

class LogTransformer(BaseEstimator, TransformerMixin):
    """Apply log1p to non-negative values. Inverse: expm1."""

    def __init__(self, shift=0.0):
        self.shift = shift  # stored as attribute so get_params() finds it

    def fit(self, X, y=None):
        return self  # stateless — nothing to learn from training data

    def transform(self, X):
        X = np.asarray(X, dtype=float)          # coerce to numpy
        return np.log1p(np.maximum(X + self.shift, 0))  # clip negatives before log

    def inverse_transform(self, X):
        return np.expm1(np.asarray(X)) - self.shift


# ── Verify the custom transformer behaves like a proper sklearn object ──

lt = LogTransformer(shift=1.0)
print("get_params() :", lt.get_params())  # should show {'shift': 1.0}

lt_clone = clone(lt)                      # clone must work for GridSearchCV
print("clone works  :", lt_clone.get_params())

sample = np.array([[0, 10, 100, 1000]])
transformed   = lt.fit_transform(sample)
reconstructed = lt.inverse_transform(transformed)
print("Original     :", sample)
print("Transformed  :", transformed.round(3))
print("Reconstructed:", reconstructed.round(3))

# Confirm usable in cross_val_score
pipe_lt = Pipeline([
    ('log',   LogTransformer()),
    ('scaler', StandardScaler()),
    ('model',  Ridge()),
])
X_raw_num = X_train[['accommodates', 'num_reviews']].fillna(0).values
scores_lt = cross_val_score(pipe_lt, X_raw_num, y_train, cv=3, scoring='r2')
print(f"\nCross-val R² with LogTransformer pipeline: {scores_lt.mean():.3f}")

In [ ]:
np.random.seed(42)

class CyclicalEncoder(BaseEstimator, TransformerMixin):
    """Encode a cyclic feature (e.g. month 1-12, hour 0-23) as sin/cos pair.

    Parameters
    ----------
    period : int
        The full cycle length (12 for months, 24 for hours, 7 for days).
    """

    def __init__(self, period=12):
        self.period = period

    def fit(self, X, y=None):
        return self  # stateless

    def transform(self, X):
        X = np.asarray(X).ravel()             # flatten to 1-D
        angle = 2 * np.pi * X / self.period   # map to [0, 2pi]
        return np.column_stack([np.sin(angle), np.cos(angle)])

    def get_feature_names_out(self, input_features=None):
        prefix = input_features[0] if input_features is not None else 'x'
        return np.array([f'{prefix}_sin', f'{prefix}_cos'])


# ── Verify CyclicalEncoder ──
months = np.array([1, 6, 12, 12, 1])  # Dec and Jan should be close in embedding space
enc = CyclicalEncoder(period=12)
encoded = enc.fit_transform(months)
print("Month  to (sin, cos)")
for m, row in zip(months, encoded):
    print(f"  {m:>2}   ({row[0]:+.3f}, {row[1]:+.3f})")
print()
print("Distance Jan to Dec :", np.linalg.norm(encoded[2] - encoded[4]).round(4))
print("Distance Jan to Jun :", np.linalg.norm(encoded[0] - encoded[1]).round(4))
print("(Jan and Dec should be much closer than Jan and Jun)")

In [ ]:
np.random.seed(42)

class DomainFeatureEngineer(BaseEstimator, TransformerMixin):
    """Create Airbnb-domain ratio and interaction features.

    Expected column order: [bedrooms, bathrooms, accommodates, review_score, num_reviews]
    """

    def fit(self, X, y=None):
        return self  # purely functional, no state needed

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        bedrooms      = X[:, 0].clip(min=1)   # avoid division by zero
        accommodates  = X[:, 2]
        review_score  = X[:, 3]
        num_reviews   = X[:, 4]

        accommodates_per_bedroom = accommodates / bedrooms
        # review_credibility: high score * many reviews = high credibility
        review_credibility = review_score * np.log1p(num_reviews)

        return np.column_stack([
            accommodates_per_bedroom,
            review_credibility,
        ])

    def get_feature_names_out(self, input_features=None):
        return np.array(['accommodates_per_bedroom', 'review_credibility'])


# ── Verify DomainFeatureEngineer ──
numeric_features = ['bedrooms', 'bathrooms', 'accommodates', 'review_score', 'num_reviews']

sample_X = np.array([
    [2, 1, 4, 4.8, 120],   # 4 guests / 2 beds = 2.0, high credibility
    [1, 1, 1, 3.0,   2],   # 1 guest / 1 bed  = 1.0, low credibility
])

dfe = DomainFeatureEngineer()
result = dfe.fit_transform(sample_X)
print("Input (bedrooms, bathrooms, accommodates, review_score, num_reviews):")
print(sample_X)
print()
print("Engineered features [accommodates_per_bedroom, review_credibility]:")
print(result.round(3))

## ColumnTransformer: Different Recipes for Different Column Types

Real datasets contain numeric columns, categorical columns, and text columns — each needing a different preprocessing recipe.

`ColumnTransformer` acts like a **routing switchboard**:
- Numeric columns go to the numeric pipeline
- Categorical columns go to the categorical pipeline
- Text column goes to TF-IDF

All branches are fitted simultaneously on the training data. Their outputs are concatenated horizontally into a single feature matrix.

```
                +-numeric pipeline---------+
Raw DataFrame --+-categorical pipeline------+---> feature matrix
                +-TF-IDF (text)------------+
```

**Key parameters:**
- `remainder='drop'` (default) — columns not listed are discarded
- `remainder='passthrough'` — unlisted columns pass through unchanged
- `n_jobs=-1` — fit branches in parallel


In [ ]:
np.random.seed(42)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Column group definitions ──
numeric_features     = ['bedrooms', 'bathrooms', 'accommodates', 'review_score', 'num_reviews']
categorical_features = ['neighbourhood_group', 'room_type']
text_feature         = 'description'

# ── Numeric branch: impute -> log -> scale ──
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # fill NaN with median
    ('log',     LogTransformer()),                  # reduce right skew
    ('scaler',  StandardScaler()),                  # zero mean, unit variance
])

# ── Categorical branch: impute -> one-hot encode ──
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # fill NaN with mode
    ('ohe',     OneHotEncoder(
        handle_unknown='ignore',   # unseen categories become all-zeros
        sparse_output=False        # return dense array (sklearn >= 1.2)
    )),
])

# ── Assemble the switchboard ──
preprocessor = ColumnTransformer(
    transformers=[
        ('num',  numeric_transformer,     numeric_features),
        ('cat',  categorical_transformer, categorical_features),
        ('text', TfidfVectorizer(max_features=50), text_feature),
    ],
    remainder='drop',
    n_jobs=-1
)

# Quick sanity-check: fit and transform training data
X_transformed_check = preprocessor.fit_transform(X_train, y_train)

print("Input shape    :", X_train.shape)
print("Output shape   :", X_transformed_check.shape)
n_ohe = preprocessor.named_transformers_['cat']['ohe'].get_feature_names_out().shape[0]
print(f"  Numeric  : {len(numeric_features)} cols -> {len(numeric_features)} features")
print(f"  Categoric: {len(categorical_features)} cols -> {n_ohe} one-hot features")
print(f"  Text     : 50 TF-IDF features")
print(f"  Total    : {X_transformed_check.shape[1]}")

In [ ]:
np.random.seed(42)

# ── Full end-to-end pipeline: preprocessor -> Ridge ──
full_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model',      Ridge(alpha=1.0)),
])

# 5-fold cross-validation with one call — leakage-free by construction
cv_scores = cross_val_score(
    full_pipeline, X_train, y_train,
    cv=5, scoring='r2', n_jobs=-1
)

print("5-fold Cross-Validation R2")
for i, s in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {s:.4f}")
print(f"  Mean : {cv_scores.mean():.4f}")
print(f"  Std  : {cv_scores.std():.4f}")

# Also fit on full training set and evaluate on held-out test set
full_pipeline.fit(X_train, y_train)
test_r2 = r2_score(y_test, full_pipeline.predict(X_test))
print(f"\nHeld-out Test R2: {test_r2:.4f}")

In [ ]:
np.random.seed(42)

# ── Visualise the pipeline structure ──
# sklearn can render a visual HTML diagram in Jupyter notebooks.
# We also show the step-by-step data shape transformation.

from sklearn import set_config
set_config(display='text')    # 'diagram' renders HTML in Jupyter; 'text' works everywhere
print("Pipeline structure:")
print(full_pipeline)
print()

# Show data shapes at each stage by accessing intermediate steps manually
print("Data shape transformations:")
print(f"  Raw X_train         : {X_train.shape}")
X_after_preprocess = full_pipeline.named_steps['preprocess'].transform(X_train)
print(f"  After preprocessor  : {X_after_preprocess.shape}")
print(f"  Ridge output        : scalar prediction per row")
print()

# Demonstrate that pipeline steps are accessible by name
ridge_coefs = full_pipeline.named_steps['model'].coef_
print(f"Ridge coefficient vector shape: {ridge_coefs.shape}")
print(f"Top-5 absolute coefficients   : {np.sort(np.abs(ridge_coefs))[-5:][::-1].round(4)}")

In [ ]:
np.random.seed(42)

from sklearn.model_selection import GridSearchCV

# Double-underscore syntax navigates the pipeline tree:
#   'preprocess__text__max_features'
#    step name --- sub-step --- parameter

param_grid = {
    'preprocess__text__max_features': [25, 50, 100],   # TfidfVectorizer param
    'model__alpha':                   [0.1, 1.0, 10.0], # Ridge regularisation
}

grid = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=0,
    refit=True,        # refit best params on full training data automatically
)
grid.fit(X_train, y_train)

print("Best parameters :", grid.best_params_)
print(f"Best CV R2      : {grid.best_score_:.4f}")
print(f"Test R2         : {r2_score(y_test, grid.predict(X_test)):.4f}")
print()
cv_results = pd.DataFrame(grid.cv_results_)
top3 = cv_results.nlargest(3, 'mean_test_score')[[
    'param_preprocess__text__max_features',
    'param_model__alpha',
    'mean_test_score',
    'std_test_score'
]]
print("Top-3 parameter combinations:")
print(top3.to_string(index=False))

In [ ]:
np.random.seed(42)

# ── Get feature names from the fitted pipeline ──
# sklearn 1.0+ supports get_feature_names_out() on ColumnTransformer
feature_names = full_pipeline[:-1].get_feature_names_out()

print(f"Total features after preprocessing: {len(feature_names)}")
print()

num_feats  = [f for f in feature_names if f.startswith('num__')]
cat_feats  = [f for f in feature_names if f.startswith('cat__')]
text_feats = [f for f in feature_names if f.startswith('text__')]

print(f"  Numeric   features ({len(num_feats)}): {num_feats}")
print(f"  Categoric features ({len(cat_feats)}): {cat_feats}")
print(f"  Text      features ({len(text_feats)}): first 5 = {text_feats[:5]}")
print()

# Show coefficient importance for numeric + categorical features
ridge_coefs = full_pipeline.named_steps['model'].coef_
coef_df = (
    pd.DataFrame({'feature': feature_names, 'coef': ridge_coefs})
    .assign(abs_coef=lambda d: d['coef'].abs())
    .query("not feature.str.startswith('text__')")
    .sort_values('abs_coef', ascending=False)
    .head(12)
)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['coef']]
ax.barh(coef_df['feature'], coef_df['coef'], color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Ridge Coefficient')
ax.set_title('Feature Coefficients (non-text features, top 12 by magnitude)')
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

import joblib
import os

# ── Serialize the fitted pipeline to disk ──
pipeline_path = 'airbnb_pipeline.pkl'
joblib.dump(full_pipeline, pipeline_path)

file_size_kb = os.path.getsize(pipeline_path) / 1024
print(f"Pipeline saved to '{pipeline_path}' ({file_size_kb:.1f} KB)")

# ── Reload and verify predictions are byte-for-byte identical ──
loaded_pipeline = joblib.load(pipeline_path)

original_preds = full_pipeline.predict(X_test[:5])
loaded_preds   = loaded_pipeline.predict(X_test[:5])

np.testing.assert_array_almost_equal(
    original_preds, loaded_preds,
    decimal=8,
)

print("Pipeline saved and loaded successfully — predictions are identical.")
print()
print("First 5 log-price predictions:")
for i, (orig, load) in enumerate(zip(original_preds, loaded_preds)):
    print(f"  Row {i}: original={orig:.6f}, loaded={load:.6f}")

In [ ]:
np.random.seed(42)

def predict_price(listing_dict, pipeline_path='airbnb_pipeline.pkl'):
    """Predict nightly price (USD) from a dictionary of listing features.

    Parameters
    ----------
    listing_dict : dict
        Keys must match the training feature columns.
    pipeline_path : str
        Path to the serialised sklearn Pipeline.

    Returns
    -------
    float
        Predicted nightly price in USD.
    """
    pipeline = joblib.load(pipeline_path)      # production: cache in memory
    df_input = pd.DataFrame([listing_dict])    # pipeline expects a DataFrame
    log_price_pred = pipeline.predict(df_input)[0]  # predict in log space
    return round(float(np.expm1(log_price_pred)), 2) # invert log to get USD


test_listings = [
    {
        'bedrooms': 2, 'bathrooms': 1.0, 'accommodates': 4,
        'review_score': 4.8, 'num_reviews': 45,
        'neighbourhood_group': 'Central', 'room_type': 'Entire home/apt',
        'description': 'Beautiful apartment with panoramic city views',
        'is_superhost': 1,
    },
    {
        'bedrooms': 1, 'bathrooms': 0.5, 'accommodates': 2,
        'review_score': 3.9, 'num_reviews': 8,
        'neighbourhood_group': 'East', 'room_type': 'Private room',
        'description': 'Budget room shared bathroom close to centre',
        'is_superhost': 0,
    },
    {
        'bedrooms': 4, 'bathrooms': 3.0, 'accommodates': 8,
        'review_score': 4.95, 'num_reviews': 210,
        'neighbourhood_group': 'Central', 'room_type': 'Entire home/apt',
        'description': 'luxury penthouse rooftop terrace skyline',
        'is_superhost': 1,
    },
]

print("Production inference demo")
print("-" * 40)
for i, listing in enumerate(test_listings, 1):
    price = predict_price(listing)
    print(f"Listing {i} ({listing['room_type']}, {listing['neighbourhood_group']}): ${price:.2f}/night")

In [ ]:
np.random.seed(42)

# ── Robustness test: malformed and edge-case inputs ──
# A production pipeline must not crash on common data quality issues.

edge_cases = [
    {
        'name': 'Missing numeric values',
        'data': {
            'bedrooms': None, 'bathrooms': None, 'accommodates': 2,
            'review_score': 4.5, 'num_reviews': 10,
            'neighbourhood_group': 'Central', 'room_type': 'Private room',
            'description': 'cosy studio near transport links',
            'is_superhost': 0,
        }
    },
    {
        'name': 'Unknown category',
        'data': {
            'bedrooms': 1, 'bathrooms': 1.0, 'accommodates': 2,
            'review_score': 4.2, 'num_reviews': 5,
            'neighbourhood_group': 'Outer Ring',   # unseen during training
            'room_type': 'Tent',                   # unseen during training
            'description': 'outdoor camping experience',
            'is_superhost': 0,
        }
    },
    {
        'name': 'Empty description',
        'data': {
            'bedrooms': 2, 'bathrooms': 1.0, 'accommodates': 4,
            'review_score': 4.0, 'num_reviews': 20,
            'neighbourhood_group': 'West', 'room_type': 'Entire home/apt',
            'description': '',                     # empty text
            'is_superhost': 1,
        }
    },
]

print("Robustness test: pipeline must not crash on edge cases")
print("-" * 55)
for case in edge_cases:
    try:
        pred = predict_price(case['data'])
        print(f"  PASS  {case['name']:35s} -> ${pred:.2f}")
    except Exception as e:
        print(f"  FAIL  {case['name']:35s} -> {type(e).__name__}: {e}")

## Pipeline Best Practices

### 1. Always fit on training data only
Pass the Pipeline (not pre-transformed data) to `cross_val_score`, `GridSearchCV`, or `fit()` on your training set. Never call `fit_transform` on the full dataset before splitting.

### 2. Use the `memory` parameter for expensive transformers
```python
from joblib import Memory
memory = Memory(location='./cache_dir', verbose=0)
pipe = Pipeline([...], memory=memory)
```
This caches the output of each intermediate step. Re-running with a different final estimator reuses the cached transform output.

### 3. `set_output(transform='pandas')` to preserve column names
sklearn 1.2 introduced a global config to make all transformers return DataFrames instead of numpy arrays.

### 4. Test your pipeline with deliberately malformed data
Unknown categories, nulls, empty strings, type mismatches — see the robustness cell above.

### 5. Version your serialised pipeline
```python
import sklearn
metadata = {'sklearn_version': sklearn.__version__, 'trained_on': '2025-01-01'}
joblib.dump({'pipeline': full_pipeline, 'metadata': metadata}, 'model_v1.pkl')
```


In [ ]:
np.random.seed(42)

# ── Demo: set_output API (sklearn 1.2+) ──

import sklearn
from packaging import version

if version.parse(sklearn.__version__) >= version.parse('1.2'):
    from sklearn import set_config
    set_config(transform_output='pandas')

    preprocessor_pandas = ColumnTransformer(
        transformers=[
            ('num',  numeric_transformer,     numeric_features),
            ('cat',  categorical_transformer, categorical_features),
        ],
        remainder='drop',
    )
    X_transformed_df = preprocessor_pandas.fit_transform(X_train, y_train)

    print(f"Output type with set_output='pandas': {type(X_transformed_df).__name__}")
    print(f"Columns (first 10): {list(X_transformed_df.columns[:10])}")
    print(f"Shape: {X_transformed_df.shape}")

    set_config(transform_output='default')
    print("\nReset transform_output to 'default' (numpy arrays)")
else:
    print(f"sklearn {sklearn.__version__} -- set_output API requires >= 1.2")
    print("Upgrade with: pip install -U scikit-learn")

In [ ]:
np.random.seed(42)

# ── Model versioning: store pipeline + metadata together ──

import sklearn
from datetime import date

versioned_artifact = {
    'pipeline':        full_pipeline,
    'sklearn_version': sklearn.__version__,
    'trained_on':      str(date.today()),
    'feature_columns': list(X_train.columns),
    'target':          'log_price (use expm1 to get USD)',
    'cv_r2':           round(cv_scores.mean(), 4),   # cv_scores from earlier cell
    'test_r2':         round(test_r2, 4),
}

versioned_path = 'airbnb_pipeline_v1_versioned.pkl'
joblib.dump(versioned_artifact, versioned_path)

# Reload and inspect metadata before using the pipeline
loaded_artifact = joblib.load(versioned_path)
print("Loaded artifact metadata:")
for k, v in loaded_artifact.items():
    if k != 'pipeline':
        print(f"  {k:20s}: {v}")
print()

# Use the pipeline from the versioned artifact
versioned_preds = loaded_artifact['pipeline'].predict(X_test[:3])
print("Sample predictions from versioned artifact:")
for i, p in enumerate(versioned_preds):
    print(f"  Row {i}: log_price={p:.4f} -> ${np.expm1(p):.2f}/night")

In [ ]:
np.random.seed(42)

# ── Pipeline performance summary: predicted vs. actual scatter + residuals ──
sns.set_theme(style='whitegrid')

y_pred_final = full_pipeline.predict(X_test)
residuals    = y_test.values - y_pred_final

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# (a) Predicted vs. Actual
ax = axes[0]
ax.scatter(y_test.values, y_pred_final, alpha=0.4, s=20, color='steelblue')
lims = [y_test.min(), y_test.max()]
ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect fit')
ax.set_xlabel('Actual log(price)')
ax.set_ylabel('Predicted log(price)')
ax.set_title(f'Predicted vs. Actual  (Test R2={r2_score(y_test, y_pred_final):.3f})')
ax.legend()

# (b) Residual histogram
ax = axes[1]
ax.hist(residuals, bins=35, color='mediumseagreen', edgecolor='white', alpha=0.85)
ax.axvline(0, color='red', linestyle='--', lw=1.5)
ax.set_xlabel('Residual (actual - predicted)')
ax.set_ylabel('Count')
ax.set_title(f'Residual Distribution  (mean={residuals.mean():.3f}, std={residuals.std():.3f})')

plt.suptitle('Full Pipeline — Test Set Diagnostics', fontweight='bold')
plt.tight_layout()
plt.show()

from sklearn.metrics import mean_absolute_error, mean_squared_error
print(f"Test R2   : {r2_score(y_test, y_pred_final):.4f}")
print(f"Test RMSE : {mean_squared_error(y_test, y_pred_final, squared=False):.4f}")
print(f"Test MAE  : {mean_absolute_error(y_test, y_pred_final):.4f}")

## Self-Check Questions

Test your understanding before moving on. Try to answer from memory before expanding.

---

**Q1.** You fit a `StandardScaler` on the entire dataset (train + test) and then run 5-fold cross-validation. Your CV R² is 0.82. A colleague who used a Pipeline reports 0.79. Whose estimate is more trustworthy for predicting real-world performance, and why?

<details><summary>Show answer</summary>

Your colleague's **0.79 is more trustworthy**. Your scaler was fitted on all 1000 rows including the validation folds. Each validation fold has already influenced the scaling statistics, which is a form of data leakage. The scaler has effectively "peeked" at the validation data, inflating the CV score. In production, truly new data won't have been seen during scaler fitting — so 0.82 is an overly optimistic estimate. The Pipeline ensures each fold fits the scaler only on its training portion, giving an honest estimate.

</details>

---

**Q2.** You want to tune `LogTransformer(shift=...)` and `Ridge(alpha=...)` simultaneously in a GridSearchCV. Your pipeline is:
```python
pipe = Pipeline([('log', LogTransformer()), ('scaler', StandardScaler()), ('model', Ridge())])
```
Write the `param_grid` dictionary.

<details><summary>Show answer</summary>

```python
param_grid = {
    'log__shift': [0.0, 0.5, 1.0],
    'model__alpha': [0.01, 0.1, 1.0, 10.0]
}
```

The syntax is `stepname__parameter`. `GridSearchCV` will try all 3 x 4 = 12 combinations.

</details>

---

**Q3.** Why do we inherit from both `BaseEstimator` and `TransformerMixin` when writing a custom transformer? What does each parent class contribute?

<details><summary>Show answer</summary>

- **`BaseEstimator`** provides `get_params()` and `set_params()` for free — required by `GridSearchCV` and `clone()`.
- **`TransformerMixin`** provides `fit_transform(X, y)` for free — calls `self.fit(X, y).transform(X)`.

Without `BaseEstimator`, your transformer cannot be tuned or cloned. Without `TransformerMixin`, you'd have to implement `fit_transform` yourself.

</details>


## Key Takeaways

1. **Always use Pipeline — it is a correctness requirement, not a convenience.** Fitting any preprocessing step on the full dataset before cross-validation introduces data leakage.

2. **Custom transformers follow a three-method contract:** `__init__` stores hyperparameters, `fit` learns from training data (or does nothing for stateless steps), `transform` applies the learned transformation.

3. **Inherit from `BaseEstimator` + `TransformerMixin`** to get `get_params()`, `set_params()`, and `fit_transform()` for free — required for GridSearchCV compatibility.

4. **`ColumnTransformer` is your mixed-type routing switchboard.** It applies different pipelines to numeric, categorical, and text columns in parallel, then concatenates the results.

5. **Tune pipeline hyperparameters with `stepname__param` syntax.** This works at arbitrary nesting depth (`preprocess__text__max_features`).

6. **`joblib.dump` / `joblib.load` serialises the entire fitted pipeline.** Store sklearn version + metrics alongside the pickle file for production traceability.

7. **Test robustness before deployment.** Unknown categories, missing values, empty strings — your pipeline should handle them gracefully rather than crashing.

---

**Next: Notebook 13 — Hands-On Lab & Milestone: Full ML Prediction System**
